# Curvas B3 — Coleta e Interpolação de Taxas Referenciais

Este notebook coleta as curvas de juros do portal da B3 via web scraping e aplica interpolação cúbica spline para gerar taxas em dias intermediários.

## Fluxo
1. Calcula o último dia útil do mês anterior
2. Coleta via Selenium as tabelas de taxas referenciais da B3:
   - **PRE x DI** — curva prefixada
   - **DI x IPCA** — curva inflação
   - **DI x IGPM** — curva IGP-M
3. Grava os dados brutos na planilha de interpolação
4. Executa interpolação cúbica spline nas três curvas e salva os resultados

> **Nota:** Os dados exibidos nas células de output são fictícios, gerados para fins de demonstração.

## 1. Dependências e Configurações

In [ ]:
# pip install pandas openpyxl selenium webdriver-manager urllib3

import os
import time
import warnings
import urllib3
from datetime import date, timedelta
from calendar import monthrange

import pandas as pd
import openpyxl
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Configuracoes — ajuste conforme o ambiente
CAMINHO_FERIADOS = "caminho/para/feriados.xlsx"
CAMINHO_PLANILHA = "caminho/para/Curvas_B3.xlsx"
ABA_INTERPOLACAO = "Interpolacao"

# URL das taxas referenciais B3
URL_B3 = (
    "https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/"
    "market-data/consultas/mercado-de-derivativos/precos-referenciais/"
    "taxas-referenciais-bm-fbovespa/"
)

os.environ['WDM_SSL_VERIFY'] = '0'
warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning)

## 2. Último Dia Útil do Mês Anterior

In [ ]:
def obter_ultimo_dia_util(caminho_feriados: str) -> str:
    """
    Retorna o ultimo dia util do mes anterior no formato DDMMYYYY.
    Considera feriados nacionais lidos de uma planilha e fins de semana.

    Args:
        caminho_feriados: caminho para o arquivo .xlsx com coluna 'Data'

    Returns:
        str: data no formato DDMMYYYY
    """
    df_feriados = pd.read_excel(caminho_feriados, engine='openpyxl')
    lista_feriados = pd.to_datetime(df_feriados['Data'], dayfirst=True).dt.date.tolist()

    hoje = date.today()
    hoje_menos_1mes = hoje - timedelta(days=30)

    # Ultimo dia do mes anterior
    ultimo_dia = hoje_menos_1mes.replace(
        day=monthrange(hoje_menos_1mes.year, hoje_menos_1mes.month)[1]
    )

    # Ajusta para o ultimo dia util
    while ultimo_dia in lista_feriados or ultimo_dia.weekday() in [5, 6]:
        ultimo_dia -= timedelta(days=1)

    return ultimo_dia.strftime('%d%m%Y')


ultimo_dia_util = obter_ultimo_dia_util(CAMINHO_FERIADOS)
print("Ultimo dia util do mes anterior:", ultimo_dia_util)

## 3. Função de Coleta via Selenium

Encapsula o processo de scraping para evitar repetição entre as três curvas.

In [ ]:
def coletar_curva_b3(data: str, xpath_opcao: str = None) -> pd.DataFrame:
    """
    Acessa o portal da B3 e coleta a tabela de taxas referenciais
    para a data informada.

    Args:
        data: data no formato DDMMYYYY
        xpath_opcao: XPath da opcao do select para selecionar a curva desejada.
                     Se None, usa a curva padrao (PRE x DI).

    Returns:
        pd.DataFrame: tabela com colunas numericas da curva coletada
    """
    servico   = Service(ChromeDriverManager().install())
    navegador = webdriver.Chrome(service=servico)
    navegador.get(URL_B3)
    navegador.maximize_window()

    # Aceita cookies
    WebDriverWait(navegador, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//*[@id="onetrust-accept-btn-handler"]'))
    ).click()

    # Entra no iframe
    iframe = WebDriverWait(navegador, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, "iframe"))
    )
    navegador.switch_to.frame(iframe)

    # Seleciona a curva desejada (se especificado)
    if xpath_opcao:
        navegador.find_element(By.XPATH, xpath_opcao).click()

    # Insere a data
    try:
        data_input = WebDriverWait(navegador, 10).until(
            EC.presence_of_element_located((By.ID, 'Data'))
        )
        data_input.click()
        data_input.clear()
        data_input.send_keys(data)
    except Exception as e:
        print(f"Erro ao inserir data: {e}")

    # Confirma a consulta
    navegador.find_element(
        By.XPATH,
        '//*[@id="divContainerIframeBmf"]/form/div/div/div[1]/div[2]/div/div[2]/button'
    ).click()

    # Extrai a tabela
    tabela = WebDriverWait(navegador, 10).until(
        EC.presence_of_element_located((By.ID, "tb_principal1"))
    )
    linhas = tabela.find_elements(By.TAG_NAME, "tr")
    dados  = [
        [col.text.strip() for col in linha.find_elements(By.TAG_NAME, "td")]
        for linha in linhas
        if linha.find_elements(By.TAG_NAME, "td")
    ]

    navegador.switch_to.default_content()
    navegador.quit()

    df = pd.DataFrame(dados)
    df = df.replace({',': '.', '%': ''}, regex=True)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


print("Funcao de coleta definida.")

## 4. Coleta das Três Curvas

In [ ]:
# PRE x DI (curva padrao — sem selecionar opcao)
df_PRE = coletar_curva_b3(ultimo_dia_util)
print(f"PRE x DI: {len(df_PRE)} linhas")
print(df_PRE.head())

In [ ]:
# DI x IPCA (opcao 9 do select)
XPATH_IPCA = '//*[@id="divContainerIframeBmf"]/form/div/div/div[1]/div[1]/div/div/select/option[9]'
df_IPCA = coletar_curva_b3(ultimo_dia_util, xpath_opcao=XPATH_IPCA)
print(f"DI x IPCA: {len(df_IPCA)} linhas")
print(df_IPCA.head())

In [ ]:
# DI x IGPM (opcao 10 do select)
XPATH_IGPM = '//*[@id="divContainerIframeBmf"]/form/div/div/div[1]/div[1]/div/div/select/option[10]'
df_IGPM = coletar_curva_b3(ultimo_dia_util, xpath_opcao=XPATH_IGPM)
print(f"DI x IGPM: {len(df_IGPM)} linhas")
print(df_IGPM.head())

## 5. Gravação dos Dados Brutos na Planilha

In [ ]:
def gravar_curva_na_planilha(df: pd.DataFrame, col_inicio: int, colunas_saida: list,
                              col_limpar_extra: int = None):
    """
    Limpa o intervalo correspondente na planilha e grava os novos dados.

    Args:
        df: DataFrame com os dados da curva
        col_inicio: coluna inicial para limpeza e escrita (1-indexed)
        colunas_saida: lista de indices de coluna para escrita (1-indexed)
        col_limpar_extra: coluna adicional a limpar (para formulas de interpolacao)
    """
    wb = openpyxl.load_workbook(CAMINHO_PLANILHA)
    ws = wb[ABA_INTERPOLACAO]

    # Limpa dados anteriores
    for row in ws.iter_rows(min_row=3, max_row=300,
                            min_col=col_inicio, max_col=col_inicio + len(colunas_saida) - 1):
        for cell in row:
            cell.value = None

    if col_limpar_extra:
        for row in ws.iter_rows(min_row=3, max_row=7897,
                                min_col=col_limpar_extra, max_col=col_limpar_extra):
            for cell in row:
                cell.value = None

    # Grava novos dados
    for i, row in enumerate(df.itertuples(index=False), start=3):
        for j, col in enumerate(colunas_saida):
            ws.cell(row=i, column=col, value=row[j])

    wb.save(CAMINHO_PLANILHA)
    print(f"Curva gravada a partir da coluna {col_inicio}.")


# PRE x DI — colunas B, C, D (2, 3, 4)
gravar_curva_na_planilha(df_PRE,  col_inicio=2,  colunas_saida=[2, 3, 4],  col_limpar_extra=7)

# DI x IPCA — colunas J, K (10, 11)
gravar_curva_na_planilha(df_IPCA, col_inicio=10, colunas_saida=[10, 11], col_limpar_extra=14)

# DI x IGPM — colunas Q, R (17, 18)
gravar_curva_na_planilha(df_IGPM, col_inicio=17, colunas_saida=[17, 18], col_limpar_extra=21)

## 6. Interpolação Cúbica Spline

Implementacao manual do algoritmo de spline cubica natural, aplicado às três curvas para
gerar taxas em todos os dias corridos do horizonte de interesse.

In [ ]:
def cubic_spline(input_col: list, output_col: list, dias_col: list) -> list:
    """
    Interpolacao cubica spline natural.

    Args:
        input_col:  lista de prazos (eixo x) dos pontos de controle
        output_col: lista de taxas (eixo y) dos pontos de controle
        dias_col:   lista de prazos para os quais se deseja interpolar

    Returns:
        list: taxas interpoladas para cada prazo em dias_col
    """
    if len(input_col) != len(output_col):
        raise ValueError("input_col e output_col devem ter o mesmo tamanho.")

    n   = len(input_col)
    xin = input_col
    yin = output_col
    u   = [0] * (n - 1)
    yt  = [0] * n

    # Forward sweep
    for i in range(1, n - 1):
        sig    = (xin[i] - xin[i - 1]) / (xin[i + 1] - xin[i - 1])
        p      = sig * yt[i - 1] + 2
        yt[i]  = (sig - 1) / p
        u[i]   = ((yin[i + 1] - yin[i]) / (xin[i + 1] - xin[i]) -
                  (yin[i] - yin[i - 1]) / (xin[i] - xin[i - 1]))
        u[i]   = (6 * u[i] / (xin[i + 1] - xin[i - 1]) - sig * u[i - 1]) / p

    # Back substitution
    yt[n - 1] = 0
    for k in range(n - 2, -1, -1):
        yt[k] = yt[k] * yt[k + 1] + u[k]

    # Avaliacao em cada ponto de dias_col
    y_values = []
    for x in dias_col:
        klo, khi = 0, n - 1
        while khi - klo > 1:
            k = (khi + klo) // 2
            if xin[k] > x:
                khi = k
            else:
                klo = k

        h = xin[khi] - xin[klo]
        if h == 0:
            raise ValueError("Prazos duplicados em input_col.")

        a = (xin[khi] - x) / h
        b = (x - xin[klo]) / h
        y = (a * yin[klo] + b * yin[khi] +
             ((a**3 - a) * yt[klo] + (b**3 - b) * yt[khi]) * (h**2) / 6)
        y_values.append(y)

    return y_values


# Le a planilha ja preenchida com os dados brutos
df_interp = pd.read_excel(CAMINHO_PLANILHA, sheet_name=ABA_INTERPOLACAO,
                          engine='openpyxl', header=1)

# Interpolacao PRE x DI
result_PRE  = cubic_spline(
    df_interp['INPUT'].dropna().tolist(),
    df_interp['OUTPUT'].dropna().tolist(),
    df_interp['Dias'].dropna().tolist()
)

# Interpolacao DI x IPCA
result_IPCA = cubic_spline(
    df_interp['INPUT IPCA'].dropna().tolist(),
    df_interp['OUTPUT IPCA'].dropna().tolist(),
    df_interp['Dias1'].dropna().tolist()
)

# Interpolacao DI x IGPM
result_IGPM = cubic_spline(
    df_interp['INPUT IGPM'].dropna().tolist(),
    df_interp['OUTPUT IGPM'].dropna().tolist(),
    df_interp['Dias2'].dropna().tolist()
)

# Grava resultados na planilha
wb    = openpyxl.load_workbook(CAMINHO_PLANILHA)
sheet = wb[ABA_INTERPOLACAO]

for row, value in zip(range(3, 3 + len(result_PRE)),  result_PRE):
    sheet[f'F{row}'] = value
for row, value in zip(range(3, 3 + len(result_IPCA)), result_IPCA):
    sheet[f'M{row}'] = value
for row, value in zip(range(3, 3 + len(result_IGPM)), result_IGPM):
    sheet[f'T{row}'] = value

wb.save(CAMINHO_PLANILHA)
print("Interpolacao concluida e resultados salvos na planilha.")